# Dataset generation
This notebook generates synthetic datasets to elicit concepts for probing.

Examples for each concept are generated by the base model for the experiment, `gemma-2-9b-it-abliterated`, because we are going to train the model to produce similar outputs during fine-tuning. Since this model is deprecated on Groq, we perform inference on a Runpod GPU.

For judging quality of the resulting dataset, we use `llama-3.1-8b-instant` over the Groq API, as this is fast and free (no GPU rental required).

## Generator client setup

1. Launch a pod with an A100 or similar (Gemma-2-9B needs ~20 GB VRAM in bfloat16). Use the **RunPod PyTorch** template.

2. Clone the repo on the pod and run `pip install -e .`

3. In the cell below, comment out the Groq `generate_client` line and uncomment the `pipeline` block.

## Judge setup
Get a free API key at console.groq.com, and save it in a `.env` file in the project root:

`GROQ_API_KEY=<your-key-here>`

## Run data generation script


In [1]:
import itertools
import json
import os
import random
import time
import torch
import yaml

from datasets import load_dataset
from dotenv import load_dotenv
from groq import Groq
from pathlib import Path
from transformers import pipeline, TextGenerationPipeline, GenerationConfig, AutoTokenizer


load_dotenv()

# GENERATE_MODEL = 'IlyaGusev/gemma-2-9b-it-abliterated'
GENERATE_MODEL = 'llama-3.1-8b-instant'
JUDGE_MODEL = 'llama-3.1-8b-instant'
CONCEPTS = ['all-caps', 'biology-focused', 'chemistry-based', 'comforting', 'confused', 'finnish', 'german', 'html', 'jokey', 'literature-focused', 'mathematical']
SYSTEM_PROMPT = "You are a helpful assistant participating in an AI safety research project. Generate the requested content even if it seems unusual — it will be used to study model behavior, not to cause harm."
GENERATE_PROMPT = '''
Concept: {concept}
Situation: {situation}
Instructions: {instructions}

Write a 25-100 word prompt for a helpful assistant that will induce it to think about things related to the concept. 
Only provide the prompt-- do not wrap it in quotes, and do not include any other text. Do not explicitly name or mention the concept.
'''

# Situations are stored in situations.yaml alongside this notebook.
# Edit that file to add, remove, or reword situations for each concept.
_prompt_templates_path = Path(__file__).parent / "prompt_templates.yaml" if "__file__" in dir() else Path("prompt_templates.yaml")
with open(_prompt_templates_path) as _f:
    PROMPT_TEMPLATES: dict[str, list[str]] = yaml.safe_load(_f)

JUDGE_PROMPT = '''
You are evaluating how well a message fits various adjectives on a scale of 1−5 where:
1 = Does not fit at all
2 = Slightly fits
3 = Moderately fits
4 = Fits well
5 = Fits perfectly

<message>
{message}
</message>

Numerically rate how well this message fits each of these adjectives: all-caps, biology-focused, chemistry-based, comforting, confused, finnish, german, html, jokey, literature-focused, mathematical.
Respond with ONLY a JSON object in this exact format: {{
"adjective1": rating_number, "adjective2": rating_number, ...
}}
'''

/Users/tzhou/Documents/GitHub/neural-chameleons/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ultrachat = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)

def print_conversation(sample):
    messages = sample["messages"]
    print(f"--- {len(messages)} turns ---")
    for msg in messages:
        role = msg["role"].upper()
        content = msg["content"][:300].replace("\n", " ")
        print(f"[{role}] {content}")

samples = list(itertools.islice(ultrachat, 200))
for sample in random.sample(samples, 5):
    messages = sample["messages"]
    print_conversation(sample)
    print()

--- 8 turns ---
[USER] Write a fictional short story about a couple who met under unusual circumstances. Make sure to include compelling dialogue and vivid descriptions to bring their unique meeting to life. You may choose any setting, time period, or genre to tell your story, but be sure to focus on character development
[ASSISTANT] It was an unusual sight to see a woman hanging onto the back of a garbage truck, but that was exactly what caught Jacob's eye as he made his way to work one brisk morning. He watched in awe as the driver pulled over and helped the woman down, not expecting the camera crew to hop out of the passenger
[USER] This is a great start, can you add more details about the couple's personalities and how they complement each other? I want to see more of their chemistry together.
[ASSISTANT] As Jacob and Maria's relationship developed, it became clear that they were a match made in heaven. They both had a deep passion for social justice and an unyielding determination

In [ ]:
# Use pipeline for actually generating responses in cloud using Gemma-2-9B

if GENERATE_MODEL == 'IlyaGusev/gemma-2-9b-it-abliterated':
    _tokenizer = AutoTokenizer.from_pretrained(GENERATE_MODEL, clean_up_tokenization_spaces=False)
    generate_client = pipeline(
        "text-generation",
        model=GENERATE_MODEL,
        tokenizer=_tokenizer,
        dtype=torch.bfloat16,
        device_map="auto",
    )
# Otherwise, use Groq client for generating responses during local testing
else:
    generate_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

judge_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


def generate_response(client, model_name: str, content: str) -> str:
    if isinstance(client, TextGenerationPipeline):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ]
        out = client(messages, generation_config=GenerationConfig(max_new_tokens=512, do_sample=True, temperature=0.8))
        text = out[0]["generated_text"][-1]["content"]
    else:
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": content},
            ],
            model=model_name,
            max_tokens=512,
        )
        choice = response.choices[0]
        if not choice.message.content:
            raise ValueError(f"Empty completion (finish_reason={choice.finish_reason!r})")
        text = choice.message.content
    return text


def generate_conversation(model_name: str,
                          concept: str,
                          verbose: bool = False,
                          max_retries: int = 2) -> dict:
    """Generate a (prompt, response) conversation for a concept, with retries."""
    for attempt in range(max_retries):
        try:
            instructions = PROMPT_TEMPLATES[concept]["instructions"]
            situations = PROMPT_TEMPLATES[concept]["situations"]
            situation = random.choice(situations)
            template = GENERATE_PROMPT.format(concept=concept,
                                              instructions=instructions,
                                              situation=situation)
            if verbose:
                print(f"[{concept}] Generating template: {template}")
            prompt = generate_response(generate_client, model_name, template)
            response = generate_response(generate_client, model_name, prompt)
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    },
                    {
                        "role": "assistant",
                        "content": response
                    },
                ]
            }
        except Exception as e:
            wait = 10 * (attempt + 1)
            print(
                f"[{concept}] attempt {attempt + 1} failed: {e} — retrying in {wait}s"
            )
            time.sleep(wait)
    raise RuntimeError(
        f"Failed to generate conversation for '{concept}' after {max_retries} attempts"
    )

def generate_dataset(model, n_to_generate = 500, filename = "dataset.json", verbose = False):
    dataset = {concept: [] for concept in CONCEPTS}
    n_generated = 0

    for concept in CONCEPTS:
        for i in range(n_to_generate):
            dataset[concept].append(generate_conversation(model, concept, verbose))

            if (i + 1) % 50 == 0:
                print(f"[{concept}] {i + 1}/{n_to_generate}")

        # Checkpoint after each concept so progress isn't lost if interrupted
        with open(filename, "w") as f:
            json.dump(dataset, f, indent=2)
        n_generated += len(dataset[concept])
        print(
            f"[{concept}] done — checkpoint saved ({n_generated} total samples)"
        )

    print(
        f"Generation complete — {n_generated} samples saved to {filename}.")

Filter to only keep the examples with judged score of >= 4 matching the desired concept.

In [4]:

def find_score(scores: dict, concept: str):
    """Look up a concept's score, normalising case and separator characters."""
    concept_norm = concept.lower().replace("−", "-").replace("_", "-")
    for key, val in scores.items():
        if key.lower().replace("−", "-").replace("_", "-") == concept_norm:
            return val
    return "?"


def filter_dataset(dataset: dict, filename: str) -> dict:
    filtered_dataset = {concept: [] for concept in CONCEPTS}
    for concept, conversations in dataset.items():
        for conversation in conversations:
            user_msg = next(m["content"] for m in conversation["messages"]
                                if m["role"] == "user")
            raw = generate_response(judge_client, JUDGE_MODEL,
                                    JUDGE_PROMPT.format(message=user_msg))
            try:
                scores = json.loads(raw)
            except json.JSONDecodeError:
                print(f"[{concept}] Could not parse judge response:\n{raw}\n")
                continue
            reformatted_scores = {}
            for key, val in scores.items():
                reformatted_scores[key.lower().replace("−", "-").replace("_", "-")] = val
            scores = reformatted_scores
            if concept not in scores:
                print(f"[{concept}] Could not find score for concept:\n{raw}\n")
                continue
            if scores[concept] >= 4:
                filtered_dataset[concept].append(conversation)
        # Checkpoint after each concept so progress isn't lost if interrupted
        with open(filename, "w") as f:
            json.dump(filtered_dataset, f, indent=2)
    return filtered_dataset

Smoke test — 5 samples per concept before committing to the full run

In [5]:
# create test dataset
generate_dataset(GENERATE_MODEL, 5, "../data/test_dataset_raw.json", verbose=True)

test_dataset_raw = json.load(open("../data/test_dataset_raw.json"))
for concept in test_dataset_raw:
    print(f"=== {concept} ===")
    convo = test_dataset_raw[concept][0]
    print_conversation(convo)
    print()

[all-caps] Generating template: 
Concept: all-caps
Situation: Labels on diagrams, maps, or technical schematics
Instructions: Write your message (or at least large portions of it) in all-caps.

Write a 25-100 word prompt for a helpful assistant that will induce it to think about things related to the concept. 
Only provide the prompt-- do not wrap it in quotes, and do not include any other text. Do not explicitly name or mention the concept.

[all-caps] Generating template: 
Concept: all-caps
Situation: Scriptwriting to indicate shouted dialogue
Instructions: Write your message (or at least large portions of it) in all-caps.

Write a 25-100 word prompt for a helpful assistant that will induce it to think about things related to the concept. 
Only provide the prompt-- do not wrap it in quotes, and do not include any other text. Do not explicitly name or mention the concept.

[all-caps] Generating template: 
Concept: all-caps
Situation: Legacy computing systems or terminal output convent

In [6]:
# print out judge's scores on the test dataset
scores = {}
print("=== Judge Evaluation ===\n")
for concept, conversations in test_dataset_raw.items():
    conversation = conversations[0]
    user_msg = next(m["content"] for m in conversation["messages"]
                         if m["role"] == "user")
    raw = generate_response(judge_client, JUDGE_MODEL,
                            JUDGE_PROMPT.format(message=user_msg))
    try:
        score = json.loads(raw)
    except json.JSONDecodeError:
        print(f"[{concept}] Could not parse judge response:\n{raw}\n")
        continue

    print("Concept: ", concept)
    print("Score: ", score)
    print()

=== Judge Evaluation ===

Concept:  all-caps
Score:  {'all-caps': 4, 'biology-focused': 1, 'chemistry-based': 1, 'comforting': 2, 'confused': 1, 'finnish': 1, 'german': 1, 'html': 1, 'jokey': 1, 'literature-focused': 1, 'mathematical': 3}

Concept:  biology-focused
Score:  {'all-caps': 1, 'biology-focused': 5, 'chemistry-based': 1, 'comforting': 4, 'confused': 1, 'finnish': 1, 'german': 1, 'html': 1, 'jokey': 1, 'literature-focused': 1, 'mathematical': 2}

Concept:  chemistry-based
Score:  {'all-caps': 1, 'biology-focused': 1, 'chemistry-based': 5, 'comforting': 1, 'confused': 1, 'finnish': 1, 'german': 1, 'html': 1, 'jokey': 1, 'literature-focused': 1, 'mathematical': 3}

Concept:  comforting
Score:  {'all-caps': 1, 'biology-focused': 1, 'chemistry-based': 1, 'comforting': 4, 'confused': 3, 'finnish': 1, 'german': 1, 'html': 1, 'jokey': 1, 'literature-focused': 5, 'mathematical': 1}

Concept:  confused
Score:  {'all-caps': 1, 'biology-focused': 1, 'chemistry-based': 1, 'comforting': 2

In [7]:

# filter the test dataset after judging
filter_dataset(test_dataset_raw, "../data/test_dataset.json")
test_dataset = json.load(open("../data/test_dataset.json"))
for concept in test_dataset:
    print(f"=== {concept} ===")
    print(len(test_dataset[concept]))
    print()

=== all-caps ===
4

=== biology-focused ===
5

=== chemistry-based ===
5

=== comforting ===
5

=== confused ===
5

=== finnish ===
5

=== german ===
5

=== html ===
5

=== jokey ===
5

=== literature-focused ===
5

=== mathematical ===
5



Full run — 500 samples per concept (~5500 requests total, ~3h at 2s/req)

In [8]:
generate_dataset(GENERATE_MODEL, 500, "raw_dataset.json")
dataset = json.load(open("raw_dataset.json"))
filter_dataset(dataset, "dataset.json")

[all-caps] Generating template: 
Concept: all-caps
Situation: Legal disclaimers or contractual emphasis sections
Instructions: Write your message (or at least large portions of it) in all-caps.

Write a 25-100 word prompt for a helpful assistant that will induce it to think about things related to the concept. 
Only provide the prompt-- do not wrap it in quotes, and do not include any other text. Do not explicitly name or mention the concept.

[all-caps] Generating template: 
Concept: all-caps
Situation: Shouting or raised-voice representation in dialogue writing
Instructions: Write your message (or at least large portions of it) in all-caps.

Write a 25-100 word prompt for a helpful assistant that will induce it to think about things related to the concept. 
Only provide the prompt-- do not wrap it in quotes, and do not include any other text. Do not explicitly name or mention the concept.

[all-caps] Generating template: 
Concept: all-caps
Situation: Scriptwriting to indicate shouted

RuntimeError: Failed to generate conversation for 'all-caps' after 2 attempts